## 0. Imports and data loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
import re

In [ ]:
folder_path = r'..\data\raw'
file_type = '/*csv'

files = glob.glob(folder_path + file_type)

latest_file = max(files, key=os.path.getctime)

time = latest_file[12:].split(sep="_")

access_time = pd.Timestamp(
    year=int(time[0][0:4]),
    month=int(time[0][4:6]),
    day=int(time[0][6:8]),
    hour=int(time[1][0:2]),
    minute=int(time[1][2:4]),
    tz='Europe/Moscow'
)

df = pd.read_csv(latest_file)

df.head()

In [ ]:
print(access_time)

## 1. Dataset overview

In [ ]:
df = df[df['price_byn'] > 10]

After a log-transform the distribution becomes bell-shaped, which confirms that prices follow a **log-normal distribution**. We'll train the model on `log1p(price)` and invert the prediction back.

In [ ]:
sns.histplot(data=df[df['price_byn'] < 10000], x = 'price_byn')

plt.title('Laptop price distribution')

plt.show()

plt.savefig('distribution.png', dpi=300, bbox_inches='tight')

In [ ]:
fig, axes = plt.subplots(1, 2)
df['price_byn'].hist(ax=axes[0])
np.log(df['price_byn']).hist(ax=axes[1])

plt.savefig('logarifmization.png', dpi=300, bbox_inches='tight')

**Lots of missing values.** Out of 11 053 listings only 725 have every field filled in. The brand of the video card is the most frequently missing — although it's an important feature, we may consider dropping it entirely or recovering it from the listing subject text.

In [ ]:
print(f"Shape: {df.shape}")
print(f"Shape: {df.shape}")
print("\n==================================\n")
print(df.isna().sum())
print("\n==================================\n")
print(df.info())
print("\n==================================\n")
print(df.describe())

In [ ]:
print('Listings with all fields populated:', df.dropna().shape[0], 'of', df.shape[0])

### Unique value counts per column

In [ ]:
df.nunique()

### Geographic distribution of listings

In [ ]:
sns.histplot(data=df, x='region')
plt.title('Regions distribution')
plt.xticks(rotation=90)
plt.show()
plt.savefig('region_distribution.png', dpi=300, bbox_inches='tight')

In [ ]:
sns.histplot(data=df, x='company_ad')
plt.title('Private seller vs. company')
plt.xticks(rotation=90)
plt.show()
plt.savefig('companies.png', dpi=300, bbox_inches='tight')

### Most popular video cards

In [ ]:
plt.figure(figsize=(12, 16))

order = df['videocard_brand'].value_counts().index

ax = sns.countplot(data=df, y='videocard_brand', order=order)

plt.title('Video cards', fontsize=20, pad=20)
plt.xlabel('Count', fontsize=14)
plt.ylabel('GPU brand', fontsize=14)

ax.tick_params(axis='y', labelsize=10)
ax.tick_params(axis='x', labelsize=12)

plt.tight_layout()
plt.savefig('topbrends.png', dpi=300, bbox_inches='tight')

Many sellers tag integrated graphics as a discrete card. This needs to be cleaned up downstream.

### Display matrix types

In [ ]:
brand_order = df['matrix_type'].value_counts().index
sns.countplot(data=df, x='matrix_type', order=brand_order)
plt.title('Matrix type')
plt.xticks(rotation=90)
plt.show()

brand_order = df['display_resolution'].value_counts().index
sns.countplot(data=df, x='display_resolution', order=brand_order)
plt.title('Brands distribution')
plt.xticks(rotation=90)
plt.show()
plt.savefig('display', dpi=300, bbox_inches='tight')

Too many distinct resolution strings. It probably makes sense to engineer a synthetic feature (e.g. pixel density) or to collapse non-standard resolutions into the nearest common one — for example, mapping `1920×945` to `1920×1080`.

## 2. Missing-value handling and data cleaning

In [ ]:
print(df.nunique())
print('==========================================')
print('Unique values per column')
print(f'1. os: {df['os'].unique()}')
print(f'2. videocard: {df['videocard'].unique()}')
print(f'3. processor: {df['processor'].unique()}')
print(f'4. rom_type: {df['rom_type'].unique()}')
print(f'5. diagonal: {df['diagonal'].unique()}')
print(f'6. ram_volume: {df['ram_volume'].unique()}')
print(f'7. ram_type: {df['ram_type'].unique()}')
print(f'8. battery_life: {df['battery_life'].unique()}')
print(f'9. company_ad: {df['company_ad'].unique()}')

In [ ]:
df['videocard_brand'].unique()

In [ ]:
df['os'] = df['os'].replace({'OS X': 'Mac OS'}) #OS X is Mac OS

In [ ]:
df['company_ad'] = df['company_ad'].astype(int)

In [ ]:
## import re

def extract_gpu(subject: str):
    """Return 0 for integrated GPU, 1 for discrete"""
    if pd.isna(subject) or not isinstance(subject, str):
        return float('nan')
    s = subject

    # Integrated
    if re.search(r'mac\s*book', s, re.IGNORECASE):
        return 0
    if re.search(r'Intel\s+Iris\s+Xe', s, re.IGNORECASE):
        return 0
    if re.search(r'Intel\s+UHD\s+Graphics', s, re.IGNORECASE):
        return 0

    # Discrete
    if re.search(r'(?:NVIDIA|GeForce|RTX|GTX|MX|Quadro|Arc|Radeon)', s, re.IGNORECASE):
        return 1

    return float('nan')



def extract_gpu_model(subject: str):
    
    if pd.isna(subject) or not isinstance(subject, str):
        return float('nan')
    s = subject    
    if pd.isna(subject):
        return float('nan')

    if pd.isna(subject) or not isinstance(subject, str):
        return float('nan')
    s = subject

    def find_mem(text: str):
        """
        Pick up VRAM that immediately follows the GPU model.
        If another size follows immediately (16GB 1000GB), that's RAM+SSD — skip.
        Anything above 64 cannot be VRAM in a laptop — discard.
        """
        m = re.search(r'(\d+)\s*(?:ГБ|GB)', text, re.IGNORECASE)
        if not m:
            return float('nan')
        if int(m.group(1)) > 64:
            return float('nan')
        tail = text[m.end():m.end() + 25]
        if re.search(r'\d+\s*(?:ГБ|GB|ТБ|TB)', tail, re.IGNORECASE):
            return float('nan')
        return m.group(0)

    def fmt(base: str, mem: str):
        return f"{base} {mem}" if mem else base

    SEP = r'[\s\-]*'   # separator: space, - or nothing

    if re.search(r'mac\s*book', s, re.IGNORECASE):
        return 0
    # Intel 
    m = re.search(r'Intel\s+Iris\s+Xe\s+Graphics\s+(G\d+)', s, re.IGNORECASE)
    if m:
        return f"Intel Iris XE Graphics {m.group(1).upper()}"
    if re.search(r'Intel\s+Iris\s+Xe', s, re.IGNORECASE):
        return 'Intel Iris XE Graphics'
    if re.search(r'Intel\s+UHD\s+Graphics', s, re.IGNORECASE):
        return 'Intel UHD Graphics'
    m = re.search(r'(?:Intel\s+)?Arc\s+(A\d{3}M?)', s, re.IGNORECASE)
    if m:
        return fmt(f"Intel Arc {m.group(1).upper()}", find_mem(s[m.end():]))

    # NVIDIA Quadro 
    m = re.search(rf'(?:NVIDIA\s+)?Quadro{SEP}RTX{SEP}(\d{{3,5}})\b', s, re.IGNORECASE)
    if m:
        return fmt(f"NVIDIA Quadro RTX {m.group(1)}", find_mem(s[m.end():]))

    m = re.search(rf'(?:NVIDIA\s+)?Quadro{SEP}T{SEP}(\d{{3,4}})\s*(Max-Q)?', s, re.IGNORECASE)
    if m:
        maxq = " Max-Q" if m.group(2) else ""
        return fmt(f"NVIDIA Quadro T{m.group(1)}{maxq}", find_mem(s[m.end():]))

    m = re.search(rf'(?:NVIDIA\s+)?Quadro{SEP}P{SEP}(\d{{3,4}})\b', s, re.IGNORECASE)
    if m:
        return fmt(f"NVIDIA Quadro P{m.group(1)}", find_mem(s[m.end():]))

    m = re.search(rf'(?:NVIDIA\s+)?Quadro{SEP}M{SEP}(\d{{3,4}})\b', s, re.IGNORECASE)
    if m:
        return fmt(f"NVIDIA Quadro M{m.group(1)}", find_mem(s[m.end():]))

    # NVIDIA GeForce RTX / GTX / MX 
    m = re.search(
        rf'(?:NVIDIA\s+)?(?:GeForce\s+)?RTX{SEP}(\d{{4}}){SEP}(Ti\b)?\s*(Max-Q)?',
        s, re.IGNORECASE
    )
    if m:
        ti   = " Ti"    if m.group(2) else ""
        maxq = " Max-Q" if m.group(3) else ""
        return fmt(f"NVIDIA GeForce RTX {m.group(1)}{ti}{maxq}", find_mem(s[m.end():]))

    m = re.search(
        rf'(?:NVIDIA\s+)?(?:GeForce\s+)?GTX{SEP}(\d{{4}}){SEP}(Ti\b)?\s*(Max-Q)?',
        s, re.IGNORECASE
    )
    if m:
        ti   = " Ti"    if m.group(2) else ""
        maxq = " Max-Q" if m.group(3) else ""
        return fmt(f"NVIDIA GeForce GTX {m.group(1)}{ti}{maxq}", find_mem(s[m.end():]))

    m = re.search(rf'(?:NVIDIA\s+)?(?:GeForce\s+)?MX{SEP}(\d{{3,4}})', s, re.IGNORECASE)
    if m:
        return fmt(f"NVIDIA GeForce MX{m.group(1)}", find_mem(s[m.end():]))

    # AMD Radeon
    m = re.search(r'(?:AMD\s+)?Radeon\s+RX\s+Vega\s+M\s+GL', s, re.IGNORECASE)
    if m:
        return fmt("AMD Radeon RX Vega M GL", find_mem(s[m.end():]))

    m = re.search(rf'(?:AMD\s+)?Radeon\s+Pro{SEP}(\d{{3,4}}[A-Z]*)', s, re.IGNORECASE)
    if m:
        return fmt(f"AMD Radeon Pro {m.group(1)}", find_mem(s[m.end():]))

    m = re.search(rf'(?:AMD\s+)?(?:Radeon\s+)?RX{SEP}(\d{{3,4}}M?)', s, re.IGNORECASE)
    if m:
        return fmt(f"AMD Radeon RX {m.group(1).upper()}", find_mem(s[m.end():]))

    m = re.search(rf'(?:AMD\s+)?Radeon{SEP}(\d{{3,4}}[A-Z]*)', s, re.IGNORECASE)
    if m:
        return fmt(f"AMD Radeon {m.group(1).upper()}", find_mem(s[m.end():]))
    return(float('nan'))

In [ ]:
def extract_ram(subject):
    """Extract RAM size as a string of the form '<N> ГБ'."""
    if pd.isna(subject) or not isinstance(subject, str):
        return float('nan')
    s = subject

    # 1. Explicit RAM/ОЗУ marker.
    m = re.search(r'(\d+)\s*(?:ГБ|GB)\s*(?:RAM|ОЗУ)', s, re.IGNORECASE)
    if m:
        return f"{m.group(1)} ГБ"
    m = re.search(r'(?:RAM|ОЗУ)[\s:\-]*(\d+)\s*(?:ГБ|GB)?', s, re.IGNORECASE)
    if m:
        return f"{m.group(1)} ГБ"

    # 2. Apple-style "16/256" — first number is RAM, second is storage.
    m = re.search(r'\b(\d{1,3})\s*/\s*(\d{2,4})\b', s)
    if m:
        ram, rom = int(m.group(1)), int(m.group(2))
        if ram in {2,3,4,6,8,12,16,24,32,48,64,128} and rom >= 64:
            return f"{ram} ГБ"

    # 3. "16GB/512GB" — first value is RAM.
    m = re.search(r'(\d+)\s*(?:ГБ|GB)\s*/\s*\d+\s*(?:ГБ|GB|ТБ|TB)', s, re.IGNORECASE)
    if m and int(m.group(1)) <= 128:
        return f"{m.group(1)} ГБ"

    # 4. "16ГБ 1000ГБ" — two consecutive sizes, first one is RAM.
    m = re.search(r'(\d+)\s*(?:ГБ|GB)\s+(\d+)\s*(?:ГБ|GB|ТБ|TB)', s, re.IGNORECASE)
    if m and int(m.group(1)) <= 128:
        return f"{m.group(1)} ГБ"

    return float('nan')


def extract_rom(subject):
    """Extract storage size as a string of the form '<N> ГБ'."""
    if pd.isna(subject) or not isinstance(subject, str):
        return float('nan')
    s = subject

    # 1. Terabytes — convert to GB.
    m = re.search(r'(\d+)\s*(?:ТБ|TB)', s, re.IGNORECASE)
    if m:
        return f"{int(m.group(1)) * 1000} ГБ"

    # 2. Explicit SSD/HDD marker.
    m = re.search(r'(?:SSD|HDD)[\s\-]*(\d+)\s*(?:ГБ|GB)', s, re.IGNORECASE)
    if m:
        return f"{m.group(1)} ГБ"
    m = re.search(r'(\d+)\s*(?:ГБ|GB)\s*(?:SSD|HDD)', s, re.IGNORECASE)
    if m:
        return f"{m.group(1)} ГБ"

    # 3. Apple-style "16/256" — second number is storage.
    m = re.search(r'\b(\d{1,3})\s*/\s*(\d{2,4})\b', s)
    if m:
        ram, rom = int(m.group(1)), int(m.group(2))
        if ram in {2,3,4,6,8,12,16,24,32,48,64,128} and rom >= 64:
            return f"{rom} ГБ"

    # 4. "16GB/512GB" — second value is storage.
    m = re.search(r'\d+\s*(?:ГБ|GB)\s*/\s*(\d+)\s*(?:ГБ|GB)', s, re.IGNORECASE)
    if m and int(m.group(1)) >= 64:
        return f"{m.group(1)} ГБ"

    # 5. "16ГБ 1000ГБ" — two consecutive sizes, second one is storage.
    m = re.search(r'(\d+)\s*(?:ГБ|GB)\s+(\d+)\s*(?:ГБ|GB)', s, re.IGNORECASE)
    if m and int(m.group(1)) <= 128 and int(m.group(2)) >= 64:
        return f"{m.group(2)} ГБ"

    return float('nan')

In [ ]:
def extract_diagonal(subject):
    """Extract screen diagonal in inches (float). Valid range: 10–18."""
    if pd.isna(subject) or not isinstance(subject, str):
        return float('nan')
    s = subject

    # 1. Explicit inch marker: 15.6", 17,3'', 13.3 дюйм
    m = re.search(r'(\d{2}[,.]?\d?)\s*(?:[\'"]{1,2}|дюйм|inch)', s, re.IGNORECASE)
    if m:
        val = float(m.group(1).replace(',', '.'))
        if 10 <= val <= 18:
            return val

    # 2. Bare decimals such as 15.6, 17.3, 13.3 — rare false positives.
    m = re.search(r'\b(1[0-7][,.]\d)\b', s)
    if m:
        val = float(m.group(1).replace(',', '.'))
        if 10 <= val <= 18:
            return val

    # 3. After MacBook / Air / Pro — bare integer 13–17.
    m = re.search(r'(?:MacBook|Air|Pro)\s+(\d{2})\b', s, re.IGNORECASE)
    if m:
        val = float(m.group(1))
        if 10 <= val <= 18:
            return val

    return float('nan')

In [ ]:
def extract_processor(subject):
    """Extract a normalized processor family (e.g. Intel Core i7)."""
    if pd.isna(subject) or not isinstance(subject, str):
        return float('nan')
    s = subject

    # Apple M1/M2/M3/M4/M5 (Pro/Max/Ultra)
    m = re.search(r'\bM([1-5])\s*(Pro|Max|Ultra)?\b', s)
    # Only treat M-prefix as Apple Silicon when subject mentions Mac/Apple
    if m and re.search(r'mac\s*book|apple', s, re.IGNORECASE):
        suffix = f" {m.group(2)}" if m.group(2) else ""
        return f"Apple M{m.group(1)}{suffix}"

    # Intel Core iN
    m = re.search(r'(?:Core\s+)?i([3579])[\s\-]?\d{3,5}[A-Z]{0,2}', s, re.IGNORECASE)
    if m:
        return f"Intel Core i{m.group(1)}"
    m = re.search(r'Core\s+i([3579])', s, re.IGNORECASE)
    if m:
        return f"Intel Core i{m.group(1)}"

    # AMD Ryzen — "Ryzen 5", "R5-7520U"
    m = re.search(r'Ryzen\s+([3579])', s, re.IGNORECASE)
    if m:
        return f"AMD Ryzen {m.group(1)}"
    m = re.search(r'\bR([3579])\s*[\-–]\s*\d{3,5}[A-Z]{0,2}', s, re.IGNORECASE)
    if m:
        return f"AMD Ryzen {m.group(1)}"

    # Intel Pentium / Celeron
    if re.search(r'Pentium', s, re.IGNORECASE):
        return 'Intel Pentium'
    if re.search(r'Celeron', s, re.IGNORECASE):
        return 'Intel Celeron'

    return float('nan')

In [ ]:
def extract_brand(subject):
    if pd.isna(subject) or not isinstance(subject, str):
        return float('nan')
    s = subject

    brand_patterns = [
        (r'\bMacBook|Apple\b',                  'Apple'),
        (r'\bASUS|Aorus\b',                     'Asus'), 
        (r'\bLenovo|ThinkPad|IdeaPad|Legion\b', 'Lenovo'),
        (r'\bHP|HewlettPackard|Pavilion|EliteBook|ProBook|Omen|Victus\b', 'HP'),
        (r'\bDell|Inspiron|Latitude|XPS|Alienware|Vostro\b', 'Dell'),
        (r'\bAcer|Aspire|Predator|Nitro|Swift|TravelMate\b', 'Acer'),
        (r'\bMSI\b',                            'MSI'),
        (r'\bHuawei|MateBook\b',                'Huawei'),
        (r'\bHonor\s*MagicBook|Honor\b',        'Honor'),
        (r'\bSamsung|Galaxy\s*Book\b',          'Samsung'),
        (r'\bXiaomi|RedmiBook|Redmi\b',         'Xiaomi'),
        (r'\bGigabyte\b',                       'Gigabyte'),
        (r'\bRazer\b',                          'Razer'),
        (r'\bMicrosoft|Surface\b',              'Microsoft'),
        (r'\bLG\s*Gram|LG\b',                   'LG'),
        (r'\bToshiba|Dynabook\b',               'Toshiba'),
        (r'\bSony|VAIO\b',                      'Sony'),
        (r'\bChuwi\b',                          'Chuwi'),
        (r'\bThunderobot\b',                    'Thunderobot'),
        (r'\bMechRevo\b',                       'MechRevo'),
        (r'\bMaibenben\b',                      'Maibenben'),
        (r'\bDigma\b',                          'Digma'),
        (r'\bIRBIS\b',                          'IRBIS'),
        (r'\bHaier\b',                          'Haier'),
    ]

    for pattern, brand in brand_patterns:
        if re.search(pattern, s, re.IGNORECASE):
            return brand

    return float('nan')

Recover memory and GPU values from other columns (and from the free-text subject).

In [ ]:
df['videocard_brand'].isna().sum()

In [ ]:
print(df[['subject', 'videocard_brand']].head().to_string())
nans_before = df['videocard_brand'].isna().sum()
ram_nans_before = df['ram_volume'].isna().sum()
rom_nans_before = df['rom_volume'].isna().sum()
diag_nans_before = df['diagonal'].isna().sum()
cpu_nans_before = df['diagonal'].isna().sum()
brand_nans_before = df['brand'].isna().sum()

df['videocard'] = df['subject'].apply(extract_gpu)
df['videocard_brand'] = df['videocard_brand'].fillna(df['subject'].apply(extract_gpu_model))
df['ram_volume'] = df['ram_volume'].fillna(df['subject'].apply(extract_ram))
df['rom_volume'] = df['rom_volume'].fillna(df['subject'].apply(extract_rom))
df['diagonal'] = df['diagonal'].fillna(df['subject'].apply(extract_diagonal))
df['processor'] = df['processor'].fillna(df['subject'].apply(extract_processor))
df['brand'] = df['brand'].fillna(df['subject'].apply(extract_brand))

# Apple / Mac OS
apple_mask = (
    df['brand'].eq('Apple') |
    df['os'].eq('Mac OS')
)
df.loc[apple_mask, 'videocard'] = 0
df.loc[apple_mask, 'videocard_brand'] = 0
df.loc[apple_mask, 'rom_type'] = 'SSD'

print(df[['subject', 'videocard_brand']].head().to_string())
nans_after = df['videocard_brand'].isna().sum()
ram_nans_after = df['ram_volume'].isna().sum()
rom_nans_after = df['rom_volume'].isna().sum()
diag_nans_after = df['diagonal'].isna().sum()
cpu_nans_after = df['diagonal'].isna().sum()
brand_nans_after = df['brand'].isna().sum()

print(f'\nvideocard_brand: filled {nans_before - nans_after} of {nans_before}')
print(f'ram_volume:        filled {ram_nans_before - ram_nans_after} of {ram_nans_before}')
print(f'rom_volume:        filled {rom_nans_before - rom_nans_after} of {rom_nans_before}')
print(f'diagonal:          filled {diag_nans_before - diag_nans_after} of {diag_nans_before}')
print(f'processor:         filled {diag_nans_before - cpu_nans_after} of {cpu_nans_before}')
print(f'brand:             filled {brand_nans_before - brand_nans_after} of {brand_nans_before}')

In [ ]:
df = df.fillna(np.nan) # change None and empty strings to np.nan

In [ ]:
# if some videocard_brand -> 1
# if nan/0 ----------------> 0

videocard_nexistance_mask = (
    df['videocard_brand'].eq('0')
)
videocard_existance_mask = (
    df['videocard_brand'].notna() &
    df['videocard_brand'].ne(0) &
    df['videocard_brand'].ne('0')
)
df.loc[videocard_existance_mask, 'videocard'] = 0
df.loc[videocard_nexistance_mask, 'videocard'] = 1

In [ ]:
print(df.shape)
print(df.isna().sum())

### Format conversion

Compute the **listing age** in minutes for every ad — the difference between the scrape time and the publication time.

In [ ]:
df["list_time"] = pd.to_datetime(df["list_time"])

In [ ]:
df["timedelta_minutes"] = (access_time - df["list_time"]).dt.total_seconds() / 60

In [ ]:
df['company_ad'] = df['company_ad'].astype(int)
df.head(2)

## 3. Data preprocessing

Note how the data was collected: many fields that are semantically numeric are actually stored as strings and need explicit parsing.

In [ ]:
columns = df.columns
for column in columns:
    print(df[column].unique())

In [ ]:
def parse_number(val):
    if pd.isna(val):
        return np.nan
    match = re.search(r'\d+\.?\d*', str(val))
    return float(match.group()) if match else np.nan

def extract_first_number(series: pd.Series, transform=None) -> pd.Series:
    result = series.apply(parse_number)
    return result

df["rom_volume"] = extract_first_number(df["rom_volume"])
df["ram_volume"] = extract_first_number(df["ram_volume"])
df["diagonal"] = extract_first_number(df["diagonal"])

In [ ]:
df["diagonal"].unique()

In [ ]:
battery_map = {
    '1 час и меньше': 1,
    '1-2 часа': 1.5,
    '2-4 часа': 3,
    '4-6 часов': 5,
    '6-10 часов': 8,
    '10 и более часов' : 12,
    '1 час и меньше' : 0.5
}

condition_map = {
    "Б/у" : 0,
    "новое" : 1
}

df['battery_life'].map(battery_map)
df['condition'].map(condition_map)

## 4. Outliers and feature relationships

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 8))

# boxplot
sns.boxplot(data=df, y="price_byn", color='#CCCCCC', linewidth=0.75, ax=axes[0])
mean_price = df["price_byn"].mean()
axes[0].scatter(0, mean_price, color='red', s=50, zorder=3, label=f'Mean: {mean_price:.2f}')
axes[0].set_ylabel("Price (BYN)")
axes[0].set_title("Price distribution — boxplot")
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# log boxplot
log_price = np.log1p(df["price_byn"])
sns.boxplot(y=log_price, color='#CCCCCC', linewidth=0.75, ax=axes[1])
mean_log_price = log_price.mean()
axes[1].scatter(0, mean_log_price, color='red', s=50, zorder=3, label=f'Mean: {mean_log_price:.2f}')
axes[1].set_ylabel("log(Price + 1)")
axes[1].set_title("log(Price) — boxplot")
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

plt.savefig('boxplot.png', dpi=300, bbox_inches='tight')

On the non-log-transformed plot there are many outliers — that's the usual signature of a log-normal distribution. We keep them: they're informative for pricing the upper segment of the market.

### Price vs. listing age

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df, x="timedelta_minutes", y="price_byn",
            scatter_kws={'alpha':0.5, 's':10},
            line_kws={'color':'red', 'linewidth':2, 'linestyle':'--'})
plt.xlabel("Timedelta (min)")
plt.ylabel("Price (BYN)")
plt.title("Price vs. listing age")
plt.grid(True, alpha=0.3)
plt.show()

plt.savefig('scatter1.png', dpi=300, bbox_inches='tight')

In [ ]:
top_5_brands = df['brand'].value_counts().head(8).index.tolist()
df_top_5 = df[df['brand'].isin(top_5_brands)]

g = sns.FacetGrid(df_top_5, col="brand", col_wrap=3, height=4, sharex=True, sharey=True)
g.map(sns.scatterplot, "timedelta_minutes", "price_byn", alpha=0.5, s=10)
g.map(sns.regplot, "timedelta_minutes", "price_byn", 
      scatter=False, color='red', line_kws={'linewidth':2})
g.set_axis_labels("Timedelta (min)", "Price (BYN)")
g.set_titles(col_template="{col_name}")
g.fig.suptitle("Price vs. listing age — top 8 brands", y=1.02)
plt.tight_layout()
plt.show()

plt.savefig('scatter2.png', dpi=300, bbox_inches='tight')

Correlation with listing age is weak. The intuition: overpriced laptops simply hang around longer, so age becomes a noisy proxy for over-pricing rather than a clean signal.

In [ ]:
top_8_videocards = df['videocard_brand'].value_counts().head(9).index.tolist()
df_top_8_vc = df[df['videocard_brand'].isin(top_8_videocards)]

g = sns.FacetGrid(df_top_8_vc, col="videocard_brand", col_wrap=3, height=4, sharex=True, sharey=True)
g.map(sns.scatterplot, "timedelta_minutes", "price_byn", alpha=0.5, s=10)
g.set_axis_labels("Timedelta (min)", "Price (BYN)")
g.set_titles(col_template="{col_name}")
g.fig.suptitle("Price vs. listing age — top 8 GPUs", y=1.02)
plt.tight_layout()
plt.show()

plt.savefig('scatter3.png', dpi=300, bbox_inches='tight')

In [ ]:
top_processors = df['processor'].value_counts().head(9).index.tolist()
df_top_proc = df[df['processor'].isin(top_processors)]

g = sns.FacetGrid(df_top_proc, col="processor", col_wrap=3, height=4, sharex=True, sharey=True)
g.map(sns.scatterplot, "timedelta_minutes", "price_byn", alpha=0.5, s=10)
g.set_axis_labels("Timedelta (min)", "Price (BYN)")
g.set_titles(col_template="{col_name}")
g.fig.suptitle("Price vs. listing age — top 8 CPUs", y=1.02)
plt.tight_layout()
plt.show()

plt.savefig('scatter4.png', dpi=300, bbox_inches='tight')

As expected, **GPU and CPU set the price floor** of a laptop. Other features modulate it.

In [ ]:
sns.histplot(data=df, x='diagonal')
plt.title('Laptop diagonal distribution')

plt.savefig('diagonal.png', dpi=300, bbox_inches='tight')

In [ ]:
plt.figure(figsize=(6, 4))
sns.regplot(data=df[df['diagonal'] > 2.5], x="diagonal", y="price_byn",
            scatter_kws={'alpha':0.5, 's':10},
            line_kws={'color':'red', 'linewidth':2, 'linestyle':'-'})
plt.xlabel("Диагональ (в дюймах)")
plt.ylabel("Price (BYN)")
plt.title("Price vs. diagonal")
plt.grid(True, alpha=0.3)\

plt.savefig('scatter5.png', dpi=300, bbox_inches='tight')

In [ ]:
df = df.drop(['ad_id'], axis=1)

### Correlation matrix

In [ ]:
corr = df.select_dtypes(include="number").corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.tight_layout()
plt.show()

plt.savefig('heatmap.png', dpi=300, bbox_inches='tight')